# M2 — Entrenamiento de Segmentación de Instancias de Vehículos (YOLO11-seg)
Entrena las **5 clases obligatorias del peaje**: `moto`, `auto`, `minibus`, `camion`, `bus`.
Incluye: (A) Auto-etiquetado opcional con `YOLOE` para ampliar imágenes propias antes de corregir en Roboflow, (B) Descarga desde Roboflow y normalización de clases, (C) Entrenamiento y métricas para el informe PDF.

In [ ]:
!nvidia-smi -L || echo "Activa GPU T4 en Entorno de ejecucion > Cambiar tipo de entorno"
!pip install -q ultralytics==8.3.107 roboflow==1.1.60 pyyaml pandas matplotlib

In [ ]:
import os, glob, json, shutil, random, yaml, cv2, numpy as np, torch
from pathlib import Path
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, '| PyTorch:', torch.__version__)

# Orden canonico obligatorio de clases para el peaje
CLASES_OBJETIVO = ['moto', 'auto', 'minibus', 'camion', 'bus']
MAPA_SINONIMOS = {
    'moto': 'moto', 'motorcycle': 'moto', 'motorbike': 'moto', 'scooter': 'moto',
    'auto': 'auto', 'car': 'auto', 'sedan': 'auto', 'suv': 'auto', 'pickup': 'auto', 'hatchback': 'auto', 'vagoneta': 'auto',
    'minibus': 'minibus', 'minibús': 'minibus', 'van': 'minibus', 'minivan': 'minibus', 'trufi': 'minibus', 'microbus': 'minibus',
    'camion': 'camion', 'camión': 'camion', 'truck': 'camion', 'cargo truck': 'camion', 'trailer': 'camion', 'volqueta': 'camion',
    'bus': 'bus', 'autobus': 'bus', 'flota': 'bus', 'coach': 'bus',
}

## Parte A (Opcional) — Auto-etiquetado inicial con YOLOE para subir a Roboflow
Si tienes fotos propias sin etiquetar, súbelas en un `.zip`, genera máscaras YOLO-seg iniciales para las 5 clases y descarga `dataset_m2_prelabel.zip` para revisarlo y corregirlo en **Roboflow (Instance Segmentation)**.

In [ ]:
EJECUTAR_PREETIQUETADO = False  # Cambia a True si vas a pre-etiquetar imagenes propias crudas

if EJECUTAR_PREETIQUETADO:
    from google.colab import files
    RAW_DIR = Path('/content/m2_raw')
    DS_PRE = Path('/content/m2_prelabel')
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    shutil.rmtree(DS_PRE, ignore_errors=True)
    RAW_DIR.mkdir(parents=True)
    for fn in files.upload():
        if fn.lower().endswith('.zip'):
            !unzip -qo "$fn" -d /content/m2_raw
        else:
            shutil.move(fn, RAW_DIR / fn)

    PROMPTS_M2 = {
        'motorcycle': 'moto', 'motorbike': 'moto',
        'car': 'auto', 'sedan': 'auto', 'pickup truck': 'auto',
        'minibus': 'minibus', 'van': 'minibus', 'minivan': 'minibus',
        'truck': 'camion', 'cargo truck': 'camion',
        'bus': 'bus',
    }
    yoloe = YOLOE('yoloe-11l-seg.pt')
    nombres = list(PROMPTS_M2.keys())
    yoloe.set_classes(nombres, yoloe.get_text_pe(nombres))

    for split in ('train', 'valid'):
        (DS_PRE / split / 'images').mkdir(parents=True, exist_ok=True)
        (DS_PRE / split / 'labels').mkdir(parents=True, exist_ok=True)

    imgs = sorted(p for p in RAW_DIR.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'))
    random.seed(42); random.shuffle(imgs)
    n_val = max(1, int(len(imgs) * 0.2))
    idx_clase = {c: i for i, c in enumerate(CLASES_OBJETIVO)}

    for i, p in enumerate(imgs):
        split = 'valid' if i < n_val else 'train'
        im = cv2.imread(str(p))
        if im is None:
            continue
        H, W = im.shape[:2]
        res = yoloe.predict(im, conf=0.25, imgsz=640, device=DEVICE, agnostic_nms=True, verbose=False)[0]
        lineas = []
        if res.masks is not None and len(res.boxes):
            for poly, c in zip(res.masks.xyn, res.boxes.cls.cpu().numpy().astype(int)):
                clase = PROMPTS_M2.get(res.names[c])
                if clase is None or len(poly) < 3:
                    continue
                coords = ' '.join(f'{x:.6f} {y:.6f}' for x, y in poly)
                lineas.append(f'{idx_clase[clase]} {coords}')
        out_name = f'{i:05d}_{p.stem}.jpg'
        cv2.imwrite(str(DS_PRE / split / 'images' / out_name), im)
        (DS_PRE / split / 'labels' / f'{i:05d}_{p.stem}.txt').write_text('\n'.join(lineas))

    yaml.safe_dump({
        'path': str(DS_PRE), 'train': 'train/images', 'val': 'valid/images',
        'names': {i: c for i, c in enumerate(CLASES_OBJETIVO)}
    }, open(DS_PRE / 'data.yaml', 'w'), sort_keys=False)
    !cd /content && zip -qr dataset_m2_prelabel.zip m2_prelabel
    print('Listo: /content/dataset_m2_prelabel.zip — Súbelo a Roboflow para corregir máscaras.')

## Parte B — Descargar dataset etiquetado y aumentado desde Roboflow
Configura tu API key (mediante `getpass`, nunca la guardes fija en el notebook) y los datos de tu proyecto en Roboflow.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY') or getpass('Roboflow API Key: ')
WORKSPACE = input('Roboflow workspace slug: ').strip()
PROJECT = input('Roboflow project slug (M2): ').strip()
VERSION = int(input('Version del dataset (ej. 1): ').strip())

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov11', location='/content/dataset_m2')
DS_DIR = Path(dataset.location)
print('Dataset descargado en:', DS_DIR)

In [ ]:
# Normaliza nombres de clases del data.yaml al orden canonico: 0:moto, 1:auto, 2:minibus, 3:camion, 4:bus
yaml_path = DS_DIR / 'data.yaml'
cfg = yaml.safe_load(yaml_path.read_text())
orig_names = cfg['names']
if isinstance(orig_names, list):
    orig_names = {i: n for i, n in enumerate(orig_names)}

id_objetivo = {c: i for i, c in enumerate(CLASES_OBJETIVO)}
remap = {}
for old_id, raw_name in orig_names.items():
    norm = MAPA_SINONIMOS.get(str(raw_name).strip().lower())
    if norm is None:
        raise ValueError(f'Clase desconocida en Roboflow: {raw_name}. Agregala a MAPA_SINONIMOS.')
    remap[int(old_id)] = id_objetivo[norm]
print('Remapeo de clases:', {orig_names[k]: CLASES_OBJETIVO[v] for k, v in remap.items()})

for lbl_file in DS_DIR.rglob('labels/*.txt'):
    nuevas = []
    for linea in lbl_file.read_text().splitlines():
        partes = linea.strip().split()
        if not partes:
            continue
        old_c = int(float(partes[0]))
        if old_c in remap:
            partes[0] = str(remap[old_c])
            nuevas.append(' '.join(partes))
    lbl_file.write_text('\n'.join(nuevas))

cfg['names'] = {i: c for i, c in enumerate(CLASES_OBJETIVO)}
cfg['nc'] = len(CLASES_OBJETIVO)
yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('data.yaml normalizado con exito.')

## Parte C — Entrenamiento YOLO11-seg y exportación de métricas

In [ ]:
modelo_m2 = YOLO('yolo11s-seg.pt')
modelo_m2.train(
    data=str(DS_DIR / 'data.yaml'),
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project='/content/runs/segment',
    name='m2_vehiculos',
    exist_ok=True,
)

In [ ]:
BEST_M2 = Path('/content/runs/segment/m2_vehiculos/weights/best.pt')
val_res = YOLO(str(BEST_M2)).val(data=str(DS_DIR / 'data.yaml'))

metricas_m2 = {
    'modelo': 'yolo11s-seg',
    'clases': CLASES_OBJETIVO,
    'box_map50': round(float(val_res.box.map50), 4),
    'box_map50_95': round(float(val_res.box.map), 4),
    'mask_map50': round(float(val_res.seg.map50), 4),
    'mask_map50_95': round(float(val_res.seg.map), 4),
    'mask_map50_por_clase': {
        CLASES_OBJETIVO[i]: round(float(v), 4)
        for i, v in enumerate(val_res.seg.ap50)
        if i < len(CLASES_OBJETIVO)
    },
}
print(json.dumps(metricas_m2, indent=2, ensure_ascii=False))
Path('/content/resumen_metricas_m2.json').write_text(json.dumps(metricas_m2, indent=2, ensure_ascii=False))
shutil.copy(BEST_M2, '/content/best_m2_seg.pt')
print('Peso listo para RunPod: /content/best_m2_seg.pt')

In [ ]:
# Descargar best_m2_seg.pt y resumen_metricas_m2.json + diagnostico de entorno
import sys, platform
print('Diagnostico Colab M2:', json.dumps({
    'python': sys.version.split()[0], 'os': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))
from google.colab import files
files.download('/content/best_m2_seg.pt')
files.download('/content/resumen_metricas_m2.json')